In [0]:
# Create Gold summary by loan type
from pyspark.sql.functions import count, sum

# Connect to the Silver table (variables do not carry across notebooks)
silver_table_df = spark.table("final_project.silver.loan_repayment")

gold_df = (
    silver_table_df
    .groupBy("loan_type")
    .agg(
        count("payment_id").alias("payment_count"),
        sum("principal_amount").alias("total_principal"),
        sum("interest_amount").alias("total_interest"),
        sum("total_payment").alias("total_collections")
    )
)

display(gold_df)

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Loan_repayment.gold.repayment_summary_by_loan_type")

print("Gold table created successfully")

In [0]:

gold_table_df = spark.table(
    "Loan_repayment.gold.repayment_summary_by_loan_type"
)

display(gold_table_df.orderBy("loan_type"))

# Business Queries

Key business questions answered from the Gold layer:

1. Which loan type collects the most revenue?
2. What are the portfolio-level KPIs?
3. What is the average payment value per loan type?
4. Which loan type is most profitable per unit of principal lent?
5. Which branches collect the most?
6. What is the repayment health (payment status mix) per loan type?
7. How much of our loan book is at risk in unpaid payments?
8. How do collections trend over the month?
9. Which customers are our biggest repayers?
10. Which branches should we watch for collection risk?
11. How concentrated is our collections mix by product?

In [0]:
%sql
-- Q1: Which loan type collects the most revenue?
SELECT
    loan_type,
    payment_count,
    ROUND(total_collections, 2) AS total_collections,
    ROUND(total_collections / total_principal * 100, 2) AS effective_yield_pct
FROM loan_repayment.gold.repayment_summary_by_loan_type
ORDER BY total_collections DESC


In [0]:
%sql
-- Q2: What are the portfolio-level KPIs?
SELECT
    SUM(payment_count) AS total_payments,
    ROUND(SUM(total_principal), 2) AS total_principal_lent,
    ROUND(SUM(total_interest), 2) AS total_interest_earned,
    ROUND(SUM(total_collections), 2) AS total_collected,
    ROUND(SUM(total_interest) / SUM(total_principal) * 100, 2) AS portfolio_interest_rate_pct
FROM loan_repayment.gold.repayment_summary_by_loan_type


In [0]:
%sql
-- Q3: What is the average payment value per loan type?
SELECT
    loan_type,
    ROUND(total_collections / payment_count, 2) AS avg_payment_per_loan,
    ROUND(total_principal / payment_count, 2) AS avg_principal_per_loan
FROM loan_repayment.gold.repayment_summary_by_loan_type
ORDER BY avg_payment_per_loan DESC


In [0]:
%sql
-- Q4: Which loan type is most profitable per unit of principal lent?
SELECT
    loan_type,
    ROUND(total_interest, 2) AS total_interest_earned,
    ROUND(total_interest / total_principal * 100, 2) AS interest_yield_pct
FROM loan_repayment.gold.repayment_summary_by_loan_type
ORDER BY interest_yield_pct DESC


In [0]:
%sql
-- Q5: Which branches collect the most?
-- (Branch detail lives in the silver fact table)
SELECT
    branch_city,
    COUNT(*) AS payment_count,
    ROUND(SUM(total_payment), 2) AS total_collections
FROM final_project.silver.loan_repayment
GROUP BY branch_city
ORDER BY total_collections DESC


In [0]:
%sql
-- Q6: What is the repayment health (payment status mix) per loan type?
-- (Status detail lives in the silver fact table)
SELECT
    loan_type,
    payment_status,
    COUNT(*) AS payment_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY loan_type), 1) AS pct_of_loan_type
FROM final_project.silver.loan_repayment
GROUP BY loan_type, payment_status
ORDER BY loan_type, payment_count DESC


In [0]:
%sql
-- Q7 (Client: "How much of our loan book is at risk in unpaid payments?")
SELECT
    loan_type,
    payment_status,
    COUNT(*) AS payment_count,
    ROUND(SUM(principal_amount + interest_amount), 2) AS amount_at_risk
FROM final_project.silver.loan_repayment
WHERE payment_status IN ('PENDING', 'FAILED')
GROUP BY loan_type, payment_status
ORDER BY amount_at_risk DESC


In [0]:
%sql
-- Q8 (Client: "How do our collections trend over the month?")
SELECT
    DATE_TRUNC('WEEK', payment_date) AS payment_week,
    COUNT(*) AS payment_count,
    ROUND(SUM(total_payment), 2) AS collected_in_week
FROM final_project.silver.loan_repayment
GROUP BY DATE_TRUNC('WEEK', payment_date)
ORDER BY payment_week


In [0]:
%sql
-- Q9 (Client: "Which customers are our biggest repayers?")
SELECT
    customer_id,
    COUNT(*) AS payment_count,
    ROUND(SUM(total_payment), 2) AS total_repaid
FROM final_project.silver.loan_repayment
GROUP BY customer_id
ORDER BY total_repaid DESC
LIMIT 10


In [0]:
%sql
-- Q10 (Client: "Which branches should we watch for collection risk?")
SELECT
    branch_city,
    COUNT(*) AS failed_payments,
    ROUND(SUM(principal_amount + interest_amount), 2) AS failed_amount
FROM final_project.silver.loan_repayment
WHERE payment_status = 'FAILED'
GROUP BY branch_city
ORDER BY failed_amount DESC


In [0]:
%sql
-- Q11 (Client: "How concentrated is our collections mix by product?")
SELECT
    loan_type,
    ROUND(total_collections, 2) AS total_collections,
    ROUND(100 * total_collections / SUM(total_collections) OVER (), 1) AS pct_of_portfolio
FROM loan_repayment.gold.repayment_summary_by_loan_type
ORDER BY pct_of_portfolio DESC
